# 01 — The market tape

**The question:** *What has PETR4 actually done since 2019 — and how do I pull
seven years of daily sessions without silently losing half of them?*

That second half is the whole notebook. A naive call for seven years of daily
quotes is the single most common way to get a wrong answer out of this API, and
the reason every series function now **refuses** instead of trimming.

Endpoints: `quote_latest`, `quote_history`, and the `p_after` cursor.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
COVERAGE = as_of("quotes")

`as_of` on the `quotes` row is the last session B3 published **and** we
ingested. Every close below is at or before that date; nothing extrapolates
past it.

## One session: `quote_latest`

The wide endpoints return a fixed full row — OHLCV plus identity — and the
column list cannot vary by argument. Trim them with PostgREST's `select=` rather
than pulling twenty-two columns to read one.

In [ ]:
TICKER = "PETR4"

latest = silo.quote_latest(TICKER)[0]
for k, v in latest.items():
    print(f"  {k:18s} {v}")

Three columns there deserve attention before anything is computed from `close`:

* **`adjusted: false`.** There is no split or dividend adjustment anywhere in
  this API. `close` is the price B3 published on the day.
* **`quotation_factor`.** The published FATCOT — the number of shares the price
  refers to, `1` or `1000`. A paper quoted per lot reads a thousand times its
  unit price. `close_unit` (= `close / quotation_factor`) is the comparable one.
* **`board`.** The BDI board the row came from. `quotes` is standard-lot only.

In [ ]:
print(f"adjusted         : {latest['adjusted']}")
print(f"quotation_factor : {latest['quotation_factor']}")
print(f"close            : {latest['close']}")
print(f"close / factor   : {latest['close'] / latest['quotation_factor']}")
print()
print("CAVEAT: prices are UNADJUSTED. A 2:1 split reads as roughly -50% in")
print("close_return, with no market move behind it. This is not a total return.")

## The trap: asking for seven years in one call

`quote_history` in whole-result mode does not trim. It refuses.

In [ ]:
from silo_client.client import SiloOverCap

START = "2019-01-01"

try:
    silo.quote_history(TICKER, start=START)
except SiloOverCap as exc:
    print("refused, as it should be:")
    print(" ", exc.body)

### Why refusing is the right answer

Before catalog v26 this call returned `200` with exactly 1,000 rows — the
**oldest** 1,000 — and no signal at all. Measured against production on
2026-08-28:

```
p_from=2019-01-01  -> 1000 rows, 2019-01-02 .. 2023-01-09   TRUNCATED, HTTP 200
p_from=2022-01-01  -> 1000 rows, 2022-01-03 .. 2026-01-02   TRUNCATED, HTTP 200
p_from=2024-01-01  ->  664 rows, 2024-01-02 .. 2026-08-26   complete
```

A notebook charting "PETR4 since 2019" got a plausible line that simply stopped
in January 2023. Nothing in the response said so. **That is the failure this API
is built to make impossible** — and it is why the fix was to refuse rather than
to document the header better.

## Paging with the cursor

`quote_history` is one of three functions that hand you a cursor: send
`p_after=''` for the first page, then the last row's `trade_date` for each next
page. A page shorter than 1,000 rows is the last one.

`iter_quote_history` runs that loop.

In [ ]:
rows = silo.quote_history_all(TICKER, start=START)

px = pd.DataFrame(rows)
px["trade_date"] = pd.to_datetime(px["trade_date"])
px = px.set_index("trade_date").sort_index()

print(f"{len(px):,} sessions, {px.index.min().date()} .. {px.index.max().date()}")
print(f"pages walked: {len(px) // 1000 + 1}")
px[["open", "high", "low", "close", "volume", "adjusted"]].tail()

Compare that count against what the single call would have handed back before
v26. The difference is the part of the series you would never have known was
missing.

In [ ]:
print(f"complete series   : {len(px):,} sessions")
print(f"one page would be : 1,000 sessions "
      f"({100 * 1000 / len(px):.0f}% of it, the OLDEST rows)")
print(f"cut would have ended at: {px.index[999].date()}")
print(f"the series really ends at: {px.index[-1].date()}")

## The series, with its gaps intact

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(px.index, px["close"], linewidth=0.9)
ax.set_title(f"{TICKER} close, unadjusted — as published by B3 "
             f"(as of {COVERAGE.loc['quotes', 'as_of']})")
ax.set_ylabel("BRL (unadjusted)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print("NOT adjusted for corporate actions. Any step in this line may be a split,")
print("a grouping or a bonus rather than a move. b3_corporate_event holds the")
print("published events; no adjustment factor is served yet.")

## Returns, and the gaps the API refuses to bridge

`close_return` is a derived `panel` metric, not a `quote_history` column. Two
guarantees matter:

* A **daily** return is computed only when the previous session is within 7
  calendar days. Carnaval fits; a listing halt does not.
* A **quotation-factor change nulls the return** — a FATCOT flip rescales the
  quote by ±99.9 % with no market move behind it.

A null return is therefore *absent from the panel altogether* (the derived arm
filters nulls before emitting). Pivot locally and it becomes a NaN cell — which
is exactly what it is. **Do not fill it.**

In [ ]:
# The panel pages with the same cursor. panel_all walks it.
ret = silo.panel_all([TICKER], ["close_return"], freq="day",
                     start=START, wide=True)
ret.columns = ["close_return"]

print(f"sessions in the window     : {len(px):,}")
print(f"close_return rows emitted  : {len(ret):,}")
print(f"sessions with NO return    : {len(px) - len(ret):,}")

For `PETR4` over this window the difference is exactly **one** — the first
session, which has no predecessor to divide by. No 7-day gap, no
quotation-factor flip.

That is the honest result for a continuously-traded blue chip, and it is worth
seeing rather than assuming: the suppression rule only bites on papers that
halt, delist and relist, or change FATCOT. When it does bite, the missing rows
are **not zeros and not data to be filled** — a return across a gap is
undefined, so no row exists for it. Pivot locally and it becomes a NaN, which is
what it is.

In [ ]:
missing = px.index.difference(ret.index)
print("sessions with no close_return:")
for d in missing:
    print(" ", d.date(), "(first session in the window — nothing to divide by)")
print()
ret.describe().T

## Monthly, for mixing with fund fundamentals

`freq=day` is quotes only — no fund fundamental is daily. To put an equity next
to a fund's NAV you move to `freq=month`, where the equity value is the **last
real session of the month**, never an invented month-end bar.

In [ ]:
monthly = silo.panel([TICKER], ["close", "close_return"], freq="month",
                     start="2024-01-01", wide=True)
monthly.tail(8)

## Where this goes next

* Notebook `02` puts a fund's NAV beside a price on `freq=month`.
* Notebook `07` does the same walk for options and termo — where the series are
  short-lived, so `option_history` refuses **without** offering a cursor.
* The typed cash views (`equities`, `bdrs`, `units`, `fund_quotas`,
  `cash_securities`) split these same rows by instrument type and add a `lot`
  grain: send `lot=eq.standard` for round lots or volume double-counts.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.